# P4 — Doublons · Métrique · Coût des erreurs

| | |
|---|---|
| **Séance** | 1 |
| **Nom** | TOSSA Yoann |
| **Porteur #** | P4 |
| **Étapes Assignées** | 1.7 · 1.11 · 1.12 |

---

**Sujet 05 — Santander Satisfaction**  
**Porteur P4** · Enchaînement : 1.7 → 1.11 → 1.12

| Section | Titre | Question clé |
|---|---|---|
| 1.7 | Doublons | Y a-t-il des lignes ou colonnes identiques ? |
| 1.11 | La métrique | Sur quoi juge-t-on le modèle ? |
| 1.12 | Le coût des erreurs | Qu'est-ce qu'une erreur coûte à la banque ? |

In [ ]:
# Dépendances
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import os

In [ ]:
# Localisation des données (même logique que le carnet de démarrage)
FICHIERS_ATTENDUS = ['train.csv', 'a_decider.csv']
SUJET = "Sujet_05_Santander_Satisfaction"

def trouver_donnees(fichiers=FICHIERS_ATTENDUS, sujet=SUJET):
    pistes = []
    if os.environ.get("DONNEES_PROJET"):
        pistes.append(Path(os.environ["DONNEES_PROJET"]))
    ici = Path.cwd().resolve()
    pistes += [ici, ici / "donnees", ici / "data", ici / sujet]
    pistes += list(ici.parents)[:4]
    pistes += [Path("/kaggle/input"), Path("/content"), Path("/content/drive/MyDrive")]
    essais = []
    for racine in pistes:
        if not racine.exists():
            continue
        candidates = [racine, racine / sujet]
        if racine.name in ("input", "MyDrive"):
            candidates += [d for d in racine.iterdir() if d.is_dir()]
        for c in candidates:
            essais.append(str(c))
            if all((c / f).is_file() for f in fichiers):
                return c
    raise FileNotFoundError(
        "Données introuvables.\n"
        "Placez train.csv et a_decider.csv dans le même dossier que ce carnet,\n"
        "ou définissez DONNEES_PROJET=/chemin/vers/les/données.\n"
        "Chemins essayés : " + str(essais[:12])
    )

DOSSIER = trouver_donnees()
print("Données trouvées :", DOSSIER)

In [ ]:
# Chargement
train = pd.read_csv(DOSSIER / "train.csv")
print(f"train.csv : {train.shape[0]:,} lignes × {train.shape[1]} colonnes")

---
## 1.7 — Doublons

On cherche deux types de doublons :
- **Lignes dupliquées** : deux clients avec exactement les mêmes valeurs sur toutes les colonnes
- **Colonnes dupliquées** : deux variables qui portent exactement les mêmes valeurs

In [ ]:
# --- 1.7a : Doublons de lignes ---
n_doublons_lignes = train.duplicated().sum()
print(f"Lignes dupliquées : {n_doublons_lignes}")

if n_doublons_lignes > 0:
    print("\nExemple de doublons :")
    masque = train.duplicated(keep=False)
    display(train[masque].head(6))
else:
    print("Aucune ligne dupliquée — le jeu est propre sur ce point.")

In [ ]:
# --- 1.7b : Doublons de colonnes ---
# On compare les colonnes deux à deux (hors TARGET)
colonnes = [c for c in train.columns if c != "TARGET"]
groupes_doublons = {}

deja_vu = {}  # hash -> première colonne rencontrée
for col in colonnes:
    h = hash(train[col].values.tobytes())
    if h in deja_vu:
        ref = deja_vu[h]
        if ref not in groupes_doublons:
            groupes_doublons[ref] = []
        groupes_doublons[ref].append(col)
    else:
        deja_vu[h] = col

if groupes_doublons:
    print(f"{sum(len(v) for v in groupes_doublons.values())} colonne(s) dupliquée(s) trouvée(s) :")
    for ref, copies in groupes_doublons.items():
        print(f"  '{ref}' == {copies}")
else:
    print("Aucune colonne dupliquée.")

In [ ]:
# --- 1.7c : Bilan et décision ---
colonnes_a_supprimer = [col for copies in groupes_doublons.values() for col in copies]

print("=== BILAN 1.7 — Doublons ===")
print(f"  Lignes dupliquées     : {n_doublons_lignes}")
print(f"  Colonnes dupliquées   : {len(colonnes_a_supprimer)}")
if colonnes_a_supprimer:
    print(f"  → Ces colonnes peuvent être retirées sans perte d'information : {colonnes_a_supprimer}")
else:
    print("  → Aucune suppression nécessaire.")

---
## 1.11 — La métrique

**Pourquoi pas l'exactitude (accuracy) ?**

Avec 3,96 % de mécontents, un modèle qui prédit *toujours* « satisfait » obtient :
- Exactitude : **96,04 %** — excellent en apparence
- Mais il n'attrape **aucun** client mécontent

La métrique retenue est l'**AUC-ROC** (aire sous la courbe ROC), qui mesure la capacité du modèle à *ordonner* les clients du plus au moins risqué, indépendamment du seuil.

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

CIBLE = "TARGET"
y = train[CIBLE]

# Distribution de la cible
print("Distribution de TARGET :")
print(y.value_counts())
print(f"\nTaux de mécontents : {y.mean()*100:.2f} %")

In [ ]:
# --- Référence : modèle naïf (prédit toujours 0) ---
y_pred_naif = np.zeros(len(y))  # prédit toujours "satisfait"
exactitude_naif = (y == 0).mean()

# AUC d'un score aléatoire
np.random.seed(42)
y_pred_aleatoire = np.random.rand(len(y))
auc_aleatoire = roc_auc_score(y, y_pred_aleatoire)

print(f"Modèle naïf (prédit toujours 0) :")
print(f"  Exactitude : {exactitude_naif*100:.2f} %  ← chiffre trompeur")
print(f"  AUC        : 0.5000              ← pas mieux que le hasard")
print(f"\nModèle aléatoire :")
print(f"  AUC        : {auc_aleatoire:.4f}  ← converge vers 0.5")
print(f"\nPlancher métier (règle de la banque sans modèle) : AUC = 0.6890")

In [ ]:
# --- Visualisation : courbe ROC de référence ---
fig, ax = plt.subplots(figsize=(7, 5))

# Courbe aléatoire
ax.plot([0, 1], [0, 1], 'k--', label='Hasard (AUC = 0.50)')

# Plancher métier (ligne horizontale de référence)
ax.axhline(y=0.6890, color='orange', linestyle=':', label='Plancher métier : AUC = 0.6890')

ax.set_xlabel("Taux de faux positifs (FPR)")
ax.set_ylabel("Taux de vrais positifs (TPR)")
ax.set_title("Courbe ROC — références")
ax.legend()
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])
plt.tight_layout()
plt.savefig("../sorties/roc_reference.png", dpi=100)
plt.show()
print("Figure sauvegardée dans sorties/roc_reference.png")

In [ ]:
# --- Résumé métrique ---
print("=== BILAN 1.11 — La métrique ===")
print("  Métrique choisie : AUC-ROC")
print("  Pourquoi : l'exactitude est trompeuse sur données déséquilibrées (3.96% mécontents)")
print("  L'AUC mesure la capacité à ordonner, indépendamment du seuil de décision")
print(f"  Plancher à battre : AUC = 0.6890 (règle métier de la banque)")
print("  Un modèle utile doit dépasser ce plancher.")

---
## 1.12 — Le coût des erreurs

Deux types d'erreurs existent, avec des coûts **asymétriques** :

| Erreur | Description | Conséquence |
|---|---|---|
| **Faux négatif (FN)** | Client mécontent prédit satisfait | La banque ne rappelle pas — le client part en se plaignant |
| **Faux positif (FP)** | Client satisfait prédit mécontent | Appel inutile — coût humain, temps agent |

Un faux négatif coûte **bien plus cher** qu'un faux positif : un client perdu, une plainte, un impact réputationnel.

In [ ]:
from sklearn.metrics import confusion_matrix

# Hypothèses de coût (à ajuster selon le contexte métier)
COUT_FN = 10  # coût d'un client mécontent non rappelé (perte client + plainte)
COUT_FP = 1   # coût d'un appel inutile (temps agent)

print("Hypothèses de coût :")
print(f"  Faux Négatif (client mécontent raté) : {COUT_FN} unités")
print(f"  Faux Positif (appel inutile)         : {COUT_FP} unité")
print(f"  Ratio FN/FP : {COUT_FN}/{COUT_FP} = {COUT_FN} fois plus coûteux")

In [ ]:
# --- Simulation sur différents seuils de décision ---
# On utilise un score fictif proportionnel à la cible pour illustrer
np.random.seed(0)
# Score simulé : mécontents ont tendance à avoir un score plus élevé
score_simule = y * 0.6 + np.random.rand(len(y)) * 0.4

seuils = np.linspace(0.1, 0.9, 50)
couts = []
taux_rappel = []
taux_precision = []

for s in seuils:
    pred = (score_simule >= s).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
    cout_total = fn * COUT_FN + fp * COUT_FP
    couts.append(cout_total)
    taux_rappel.append(tp / (tp + fn) if (tp + fn) > 0 else 0)
    taux_precision.append(tp / (tp + fp) if (tp + fp) > 0 else 0)

seuil_optimal = seuils[np.argmin(couts)]
print(f"Seuil minimisant le coût total : {seuil_optimal:.2f}")
print(f"Coût minimal estimé : {min(couts):,} unités")

In [ ]:
# --- Visualisation du coût selon le seuil ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Coût total
axes[0].plot(seuils, couts, color='crimson')
axes[0].axvline(seuil_optimal, color='navy', linestyle='--', label=f'Seuil optimal : {seuil_optimal:.2f}')
axes[0].set_xlabel("Seuil de décision")
axes[0].set_ylabel("Coût total (FN×10 + FP×1)")
axes[0].set_title("Coût total selon le seuil")
axes[0].legend()

# Rappel et précision
axes[1].plot(seuils, taux_rappel, label='Rappel (clients mécontents trouvés)', color='green')
axes[1].plot(seuils, taux_precision, label='Précision (appels utiles)', color='steelblue')
axes[1].axvline(seuil_optimal, color='navy', linestyle='--', label=f'Seuil optimal : {seuil_optimal:.2f}')
axes[1].set_xlabel("Seuil de décision")
axes[1].set_ylabel("Taux")
axes[1].set_title("Rappel et précision selon le seuil")
axes[1].legend()

plt.tight_layout()
plt.savefig("../sorties/cout_erreurs.png", dpi=100)
plt.show()
print("Figure sauvegardée dans sorties/cout_erreurs.png")

In [ ]:
# --- Matrice de confusion au seuil optimal ---
pred_optimal = (score_simule >= seuil_optimal).astype(int)
tn, fp, fn, tp = confusion_matrix(y, pred_optimal).ravel()

print("=== Matrice de confusion au seuil optimal ===")
print(f"  Vrais Positifs  (TP) : {tp:,}  — mécontents bien détectés")
print(f"  Faux Positifs   (FP) : {fp:,}  — appels inutiles")
print(f"  Faux Négatifs   (FN) : {fn:,}  — mécontents ratés  ← le plus coûteux")
print(f"  Vrais Négatifs  (TN) : {tn:,}  — satisfaits bien ignorés")
print(f"\n  Coût total : {fn}×{COUT_FN} + {fp}×{COUT_FP} = {fn*COUT_FN + fp*COUT_FP:,} unités")

In [ ]:
# --- BILAN FINAL P4 ---
print("\n=== BILAN P4 — Doublons · Métrique · Coût des erreurs ===")
print()
print("1.7  Doublons")
print(f"     Lignes dupliquées  : {n_doublons_lignes}")
print(f"     Colonnes dupliquées: {len(colonnes_a_supprimer)}")
print()
print("1.11 Métrique : AUC-ROC")
print("     L'exactitude est trompeuse sur données déséquilibrées.")
print("     Plancher à battre : AUC = 0.6890")
print()
print("1.12 Coût des erreurs")
print(f"     FN (client mécontent raté) : {COUT_FN}x plus coûteux qu'un FP")
print(f"     Seuil optimal estimé       : {seuil_optimal:.2f}")
print("     → Abaisser le seuil réduit les FN mais augmente les FP.")